# task 2 -> vision-language model and visual grounding

## imports

In [ ]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

#the scripts use paths relative to the repo root (configs/, data/, results/), so the notebook runs from there
if not Path('configs').exists():
    os.chdir('..')

from common.utils import load_yaml, read_jsonl, repo_path

## global env variables ->

In [ ]:
#papermill overrides these for the smoke test, the defaults are the real run
CONFIG = 'configs/task2.yaml'
RUN_SCRIPTS = True #False -> only rebuild the tables and figures from results already on disk

In [ ]:
cfg = load_yaml(CONFIG)
RESULTS = repo_path(cfg['results_dir'])
FIGURES = RESULTS / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
DATA = repo_path(cfg['data_dir'])
#the exact configuration of this run (smoke / --scale overrides included), kept with its results
import yaml
(RESULTS / 'run_config.yaml').write_text(yaml.safe_dump(cfg, sort_keys=False))

STAGES = [s['name'] for s in cfg['stages']] #caption, instruct, vqa
STAGE_ENDS = np.cumsum([s['updates'] for s in cfg['stages']]).tolist() #2250, 3750, 4500
CONDITIONS = ['correct', 'complementary', 'mismatched', 'none']
SUITES = {'grounding_test': 'annotation-derived grounding', 'vqa_test': 'natural vqav2'}

#one fixed colour per image condition, the same in every figure
COLORS = {'correct': '#2a78d6', 'complementary': '#eb6834', 'mismatched': '#9b59b6', 'none': '#52514e'}

PYTHON = sys.executable #the interpreter running this notebook, so the scripts use the same environment

plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True,
                     'grid.color': '#e6e5e1', 'grid.linewidth': 0.6, 'axes.axisbelow': True, 'lines.linewidth': 2})


def run(command):
    #runs one script as its own process, exactly as it would be typed in a terminal. each script loads its own
    #models and writes its results to disk, nothing is passed through notebook state. finished work is skipped
    if RUN_SCRIPTS:
        get_ipython().system(command)
        assert get_ipython().user_ns['_exit_code'] == 0, f'failed -> {command}'


def load_result(name):
    return json.loads((RESULTS / name).read_text())


def stage_lines(axis):
    #dotted vertical lines at the stage boundaries (end of caption, end of instruct)
    for end in STAGE_ENDS[:-1]:
        axis.axvline(end, color='#b5b3ad', linestyle=':', linewidth=1)


def with_ci(stat, scale=100):
    return f"{scale * stat['mean']:.1f} [{scale * stat['ci_low']:.1f}, {scale * stat['ci_high']:.1f}]"

## data ->

coco 2014 captions, annotation-derived presence/counting pairs and vqav2 complementary pairs, selected with seed 6304. train2014 images for training only, val2014 images for development and final test, every evaluation set image-disjoint from the others. the downloads are cpu work, so on modal this step runs in its own cpu-only container first (`--cpu --cmd`) and is skipped here

In [ ]:
run(f'{PYTHON} -m task2_vlm.prepare_data --config {CONFIG}')
#the selection record and the selected ids live next to the splits (shared by every run on this data directory),
#copied into this run's results as the record of the data it used
selection = json.loads((DATA / 'splits' / 'done.json').read_text())
for name, source in [('selection_summary.json', 'done.json'), ('selection_ids.json', 'selection_ids.json')]:
    (RESULTS / name).write_text((DATA / 'splits' / source).read_text())
display(pd.Series({k: v for k, v in selection.items() if k not in ('types', 'rules', 'isolation')}, name='rows'))
display(pd.DataFrame(selection['types']).fillna(0).astype(int).T)
print('isolation ->', selection['isolation'], ' rules ->', selection['rules'])

## siglip feature cache ->

the frozen encoder runs once over every used image (196 patch features after siglip's final layer norm, fp16), then leaves the gpu before training

In [ ]:
run(f'{PYTHON} -m task2_vlm.cache_features --config {CONFIG}')
#one cache serves every run on this data directory, so its record lives next to the features. copied into the
#results here so each run's results directory carries the caching cost it relied on
cache = json.loads((DATA / 'feature_cache.json').read_text())
(RESULTS / 'feature_cache.json').write_text(json.dumps(cache, indent=2))
display(pd.DataFrame(cache['splits']).T)
print('caching ->', round(cache['seconds'] / 60, 2), 'min, peak memory', cache['peak_memory_gb'], 'gb on', cache['gpu'])

## training ->

one continuous run through the three stages (caption alignment, visual instruction tuning, natural vqa), development metrics before training, every 500 updates and at each stage end

In [ ]:
run(f'{PYTHON} -m task2_vlm.train --config {CONFIG}')

## final evaluation ->

both final-test suites under correct, complementary, mismatched and no-image inputs. greedy, 16 new tokens, batch size 8, thinking disabled

In [ ]:
run(f'{PYTHON} -m task2_vlm.evaluate --config {CONFIG}')

## interface checks and connector calibration

In [ ]:
checks = load_result('interface_checks.json')
display(pd.Series({k: v for k, v in checks.items() if k != 'prompt_layout'}, name='value').to_frame())
print('prompt layout ->')
print(checks['prompt_layout'])
display(pd.Series(load_result('calibration.json'), name='value').to_frame())

## learning curves ->

training loss with the stage boundaries (dotted), development caption loss, reference-answer nll under correct, complementary and no-image inputs, and the image-swap penalty dNLL = NLL(partner image) - NLL(correct image)

In [ ]:
log = pd.DataFrame(read_jsonl(RESULTS / 'train_log.jsonl'))
dev = pd.DataFrame(read_jsonl(RESULTS / 'dev_log.jsonl'))

fig, axes = plt.subplots(2, 2, figsize=(15, 9))
axis = axes[0, 0]
axis.plot(log['update'], log['loss'].rolling(25, min_periods=1).mean(), color='#0b0b0b')
axis.set_title('training loss, rolling mean over 25 updates')
axis.set_ylabel('per-example mean nll')
axis = axes[0, 1]
axis.plot(dev['update'], dev['caption_loss'], marker='o', color='#1baf7a')
axis.set_title('development caption loss')
for axis, suite in zip(axes[1], ['grounding_dev', 'vqa_dev']):
    for condition in ['correct', 'complementary', 'none']:
        axis.plot(dev['update'], dev[f'{suite}_nll_{condition}'], marker='o', color=COLORS[condition], label=condition)
    axis.set_title(f'{suite} reference-answer nll')
    axis.legend(frameon=False)
for axis in axes.flat:
    stage_lines(axis)
    axis.set_xlabel('main-run update')
fig.tight_layout()
fig.savefig(FIGURES / 'learning_curves.png', dpi=200)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(15, 3.8))
for suite, color in [('grounding_dev', '#2a78d6'), ('vqa_dev', '#eb6834')]:
    axes[0].plot(dev['update'], dev[f'{suite}_swap_penalty'], marker='o', color=color, label=suite)
axes[0].axhline(0, color='#52514e', linewidth=1)
axes[0].set_title('image-swap penalty (positive -> the correct image makes the reference more likely)')
axes[0].legend(frameon=False)
axes[1].plot(log['update'], log['lr_connector'], color='#0b0b0b', label='connector')
axes[1].plot(log['update'], log['lr_lora'], color='#e87ba4', label='lora')
axes[1].set_yscale('log')
axes[1].set_title('learning rates')
axes[1].legend(frameon=False)
for axis in axes:
    stage_lines(axis)
    axis.set_xlabel('main-run update')
fig.tight_layout()
fig.savefig(FIGURES / 'swap_penalty_and_lr.png', dpi=200)
plt.show()
dev.to_csv(RESULTS / 'dev_metrics.csv', index=False)
display(dev.round(3))

## curriculum mixture -> examples drawn per component in each stage

In [ ]:
summary = load_result('train_summary.json')
mixture = pd.DataFrame(summary['mixture_examples']).fillna(0).astype(int).T
display(pd.concat([mixture, (100 * mixture.div(mixture.sum(axis=1), axis=0)).round(1).add_suffix(' %')], axis=1))
print('updates skipped by the gradient scaler ->', summary['skipped_updates'])
print('content-cap truncations ->', summary['truncated'])

## final results ->

per suite and image condition. accuracy -> normalized exact match (grounding) or vqa soft consensus accuracy (vqav2), pair accuracy -> both questions of a pair match their own canonical answers, identical -> both predictions of a pair are the same, partner match -> a prediction equals the partner image's answer. intervals -> 95%, 4,000 bootstrap resamples of complete pairs

In [ ]:
final = load_result('eval_final.json')['summary']
tables = {}
for suite, title in SUITES.items():
    rows = []
    for c in CONDITIONS:
        s = final[suite]['conditions'][c]
        rows.append({'condition': c, 'accuracy': with_ci(s['accuracy']), 'pair_accuracy': with_ci(s['pair_accuracy']),
                     'answer_nll': with_ci(s['nll'], scale=1), 'identical_%': 100 * s['identical_rate'],
                     'empty_%': 100 * s['empty_rate'], 'partner_match_%': 100 * s['partner_match_rate'],
                     'ms_per_answer': 1000 * s['seconds_per_answer']})
    tables[suite] = pd.DataFrame(rows)
    tables[suite].to_csv(RESULTS / f'final_{suite}.csv', index=False)
    print(f"{title} -> {final[suite]['pairs']} pairs")
    display(tables[suite].round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for axis, suite in zip(axes, SUITES):
    stats = [final[suite]['conditions'][c]['accuracy'] for c in CONDITIONS]
    means = [100 * s['mean'] for s in stats]
    errors = [[100 * (s['mean'] - s['ci_low']) for s in stats], [100 * (s['ci_high'] - s['mean']) for s in stats]]
    bars = axis.bar(CONDITIONS, means, yerr=errors, capsize=4, color=[COLORS[c] for c in CONDITIONS])
    axis.bar_label(bars, fmt='%.1f', padding=6, fontsize=8)
    axis.set_title(SUITES[suite])
    axis.set_ylabel('accuracy (%)')
    axis.grid(axis='x', visible=False)
    axis.margins(y=0.15)
fig.tight_layout()
fig.savefig(FIGURES / 'final_accuracy.png', dpi=200)
plt.show()

In [ ]:
#by question type -> presence / counting (grounding) and yes/no / number / other (vqav2)
rows = []
for suite in SUITES:
    for t, by_condition in final[suite]['types'].items():
        rows.append({'suite': suite, 'type': t, **{c: with_ci(by_condition[c]['accuracy']) for c in CONDITIONS},
                     **{f'{c}_pair': with_ci(by_condition[c]['pair_accuracy']) for c in CONDITIONS}})
types = pd.DataFrame(rows)
types.to_csv(RESULTS / 'final_types.csv', index=False)
display(types)

#correct-image advantages over complementary swaps, on matched pairs -> accuracy in pp, nll in nats
advantages = pd.DataFrame([{'suite': suite, 'accuracy_advantage_pp': with_ci(final[suite]['advantages']['accuracy']),
                            'nll_advantage': with_ci(final[suite]['advantages']['nll'], scale=1)} for suite in SUITES])
advantages.to_csv(RESULTS / 'advantages.csv', index=False)
display(advantages)

## complementary-image examples ->

selection rule, fixed before looking at any output -> the first pair of every question type in the final-test file order (the seeded selection order), shown under all four image conditions. no filtering by outcome

In [ ]:
examples = pd.DataFrame(read_jsonl(RESULTS / 'eval_final_examples.jsonl'))
chosen = examples[examples['condition'] == 'correct'].groupby(['suite', 'type'], sort=False)['pair_id'].first()
for (suite, t), pid in chosen.items():
    rows = examples[examples['pair_id'] == pid]
    first = rows.iloc[0]
    print(f"\n{suite} / {t} -> {first['question']}")
    table = rows.pivot_table(index='slot', columns='condition', values='prediction', aggfunc='first')[CONDITIONS]
    table.insert(0, 'reference', rows.groupby('slot')['target'].first())
    table.insert(1, 'image', rows[rows['condition'] == 'correct'].set_index('slot')['image'])
    display(table)

In [ ]:
#the same pairs as images -> each row is one pair, each column one of its two images, titled with the reference and
#the prediction under each condition (correct = this image, complementary = the other image of the row, mismatched
#and none as listed). images come from the downloaded data, so this cell needs data/task2/images
from PIL import Image
fig, axes = plt.subplots(len(chosen), 2, figsize=(9, 3.6 * len(chosen)), squeeze=False)
for row, ((suite, t), pid) in enumerate(chosen.items()):
    rows = examples[examples['pair_id'] == pid]
    for slot in (0, 1):
        r = rows[rows['slot'] == slot].set_index('condition')
        ax = axes[row, slot]
        ax.imshow(Image.open(DATA / 'images' / f"{r.loc['correct', 'image']}.jpg").convert('RGB'))
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(False)
        preds = '\n'.join(f"{c}: {r.loc[c, 'prediction'][:40]!r}" for c in CONDITIONS)
        ax.set_title(f"{suite} / {t}\n{r.loc['correct', 'question'][:60]}\nreference: {r.loc['correct', 'target']}\n{preds}",
                     fontsize=7, loc='left')
fig.tight_layout()
fig.savefig(FIGURES / 'complementary_examples.png', dpi=200)
plt.show()

## resources

In [ ]:
evaluation = load_result('eval_final.json')
rows = [{'step': 'feature caching (siglip, once)', 'minutes': cache['seconds'] / 60, 'peak_memory_gb': cache['peak_memory_gb'],
         'trainable_params': 0}]
for stage in STAGES:
    rows.append({'step': f'training -> {stage}', 'minutes': summary['train_seconds_by_stage'][stage] / 60,
                 'peak_memory_gb': summary['peak_memory_gb'], 'trainable_params': summary['trainable_by_stage'][stage]})
rows.append({'step': 'development evaluation (all points)', 'minutes': summary['dev_seconds'] / 60,
             'peak_memory_gb': summary['dev_peak_memory_gb']})
rows.append({'step': 'checkpoint i/o', 'minutes': summary['checkpoint_seconds'] / 60})
rows.append({'step': 'final evaluation (generation + nll)', 'minutes': evaluation['seconds'] / 60})
resources = pd.DataFrame(rows)
resources.to_csv(RESULTS / 'resources.csv', index=False)
display(resources.round(3))
print(f"training -> micro batch {summary['micro_batch']} x accumulation {summary['grad_accum']}, "
      f"gradient checkpointing {summary['gradient_checkpointing']}, on {summary['gpu']}")
print('generation time per answer (ms) ->', {s: {c: round(1000 * final[s]['conditions'][c]['seconds_per_answer'], 2)
                                                  for c in CONDITIONS} for s in SUITES})